In [1]:
import numpy as np
import pandas as pd

np.random.seed(111)
n = 2500
menu_list = ["Kopi Susu", "Americano", "Cappuccino", "Roti Bakar", "Kentang Goreng", "Es Teh"]
cabang_list = ["Magelang", "Yogyakarta", "Semarang"]
waktu_list = ["Pagi", "Siang", "Sore", "Malam"]

data_tugas11 = pd.DataFrame({
    "nama_menu": np.random.choice(menu_list, size=n),
    "cabang": np.random.choice(cabang_list, size=n),
    "waktu_pesan": np.random.choice(waktu_list, size=n),
    "qty": np.random.randint(1, 6, size=n),
    "harga_satuan": np.random.choice([15000, 18000, 20000, 22000, 25000, 30000], size=n),
})
data_tugas11["total_penjualan"] = data_tugas11["qty"] * data_tugas11["harga_satuan"] + np.random.normal(0, 8000, size=n)
data_tugas11.to_csv("tugas11_penjualan.csv", index=False)
print(f"Dataset berhasil dibuat: {len(data_tugas11)} baris")

Dataset berhasil dibuat: 2500 baris


In [4]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("Tugas11").master("local[*]").getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "4")

df = spark.read.csv("tugas11_penjualan.csv", header=True, inferSchema=True)
df.printSchema()
df.show(5)

root
 |-- nama_menu: string (nullable = true)
 |-- cabang: string (nullable = true)
 |-- waktu_pesan: string (nullable = true)
 |-- qty: integer (nullable = true)
 |-- harga_satuan: integer (nullable = true)
 |-- total_penjualan: double (nullable = true)

+--------------+----------+-----------+---+------------+------------------+
|     nama_menu|    cabang|waktu_pesan|qty|harga_satuan|   total_penjualan|
+--------------+----------+-----------+---+------------+------------------+
|Kentang Goreng|  Semarang|      Malam|  1|       30000|27750.163531558544|
|Kentang Goreng|Yogyakarta|       Sore|  3|       20000| 64216.16074371864|
|Kentang Goreng|  Semarang|      Siang|  5|       22000|108722.79259154423|
|Kentang Goreng|Yogyakarta|       Pagi|  2|       20000|54483.205871792496|
|    Roti Bakar|  Magelang|      Malam|  2|       30000| 58157.94412015849|
+--------------+----------+-----------+---+------------+------------------+
only showing top 5 rows



In [5]:
from pyspark.ml.feature import StringIndexer, VectorAssembler

idx_menu_model   = StringIndexer(inputCol="nama_menu",   outputCol="nama_menu_idx").fit(df)
idx_cabang_model = StringIndexer(inputCol="cabang",      outputCol="cabang_idx").fit(df)
idx_waktu_model  = StringIndexer(inputCol="waktu_pesan", outputCol="waktu_pesan_idx").fit(df)

df_indexed = idx_menu_model.transform(df)
df_indexed = idx_cabang_model.transform(df_indexed)
df_indexed = idx_waktu_model.transform(df_indexed)

fitur_utama = ["qty", "harga_satuan", "nama_menu_idx", "cabang_idx", "waktu_pesan_idx"]
assembler = VectorAssembler(inputCols=fitur_utama, outputCol="features")
assembler_sederhana = VectorAssembler(inputCols=["qty", "harga_satuan"], outputCol="features_sederhana")

df_final = assembler.transform(df_indexed)
df_final = assembler_sederhana.transform(df_final)
df_final.select("features", "features_sederhana", "total_penjualan").show(5, truncate=False)

+-------------------------+------------------+------------------+
|features                 |features_sederhana|total_penjualan   |
+-------------------------+------------------+------------------+
|[1.0,30000.0,5.0,2.0,1.0]|[1.0,30000.0]     |27750.163531558544|
|[3.0,20000.0,5.0,1.0,3.0]|[3.0,20000.0]     |64216.16074371864 |
|[5.0,22000.0,5.0,2.0,2.0]|[5.0,22000.0]     |108722.79259154423|
|[2.0,20000.0,5.0,1.0,0.0]|[2.0,20000.0]     |54483.205871792496|
|[2.0,30000.0,4.0,0.0,1.0]|[2.0,30000.0]     |58157.94412015849 |
+-------------------------+------------------+------------------+
only showing top 5 rows



In [6]:
from pyspark.ml.regression import LinearRegression

train, test = df_final.randomSplit([0.75, 0.25], seed=101)
print("Training:", train.count(), "| Testing:", test.count())

lr = LinearRegression(featuresCol="features", labelCol="total_penjualan")
model = lr.fit(train)
print("Koefisien:", model.coefficients)
print("Intercept:", model.intercept)

Training: 1839 | Testing: 661
Koefisien: [22046.663271025758,3.022920868160672,-123.21425598657399,279.28452461677676,339.6387953231708]
Intercept: -66903.17766485889


In [7]:
from pyspark.ml.evaluation import RegressionEvaluator

predictions = model.transform(test)

rmse = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(predictions)
r2   = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(predictions)

print(f"RMSE : Rp {rmse:,.0f}")
print(f"R²   : {r2:.4f}")

RMSE : Rp 10,901
R²   : 0.9107


In [8]:
lr2 = LinearRegression(featuresCol="features_sederhana", labelCol="total_penjualan")
model2 = lr2.fit(train)
pred2 = model2.transform(test)

rmse2 = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="rmse").evaluate(pred2)
r2_2  = RegressionEvaluator(labelCol="total_penjualan", predictionCol="prediction", metricName="r2").evaluate(pred2)

hasil = pd.DataFrame({
    "Model": ["Utama (5 fitur)", "Sederhana (qty + harga_satuan)"],
    "RMSE (Rp)": [round(rmse), round(rmse2)],
    "R²": [round(r2, 4), round(r2_2, 4)],
})
print(hasil.to_string(index=False))
print(f"Selisih R² : {r2 - r2_2:+.4f}")

                         Model  RMSE (Rp)     R²
               Utama (5 fitur)      10901 0.9107
Sederhana (qty + harga_satuan)      10896 0.9108
Selisih R² : -0.0001


In [15]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

In [16]:
from pyspark.sql.functions import col

skenario = spark.sql("""
    SELECT * FROM VALUES
        ('Kopi Susu',      'Magelang',   'Pagi',  3, 22000),
        ('Cappuccino',     'Yogyakarta', 'Sore',  5, 25000),
        ('Kentang Goreng', 'Semarang',   'Malam', 2, 18000)
    AS t(nama_menu, cabang, waktu_pesan, qty, harga_satuan)
""").withColumn("qty", col("qty").cast("int")) \
   .withColumn("harga_satuan", col("harga_satuan").cast("int"))

sk = idx_menu_model.transform(skenario)
sk = idx_cabang_model.transform(sk)
sk = idx_waktu_model.transform(sk)
sk = assembler.transform(sk)

model.transform(sk).select("nama_menu", "cabang", "waktu_pesan", "qty",
                           "harga_satuan", "prediction").show(truncate=False)

+--------------+----------+-----------+---+------------+------------------+
|nama_menu     |cabang    |waktu_pesan|qty|harga_satuan|prediction        |
+--------------+----------+-----------+---+------------+------------------+
|Kopi Susu     |Magelang  |Pagi       |3  |22000       |65617.85699176662 |
|Cappuccino    |Yogyakarta|Sore       |5  |25000       |119954.93279289985|
|Kentang Goreng|Semarang  |Malam      |2  |18000       |31884.861068708575|
+--------------+----------+-----------+---+------------+------------------+



In [17]:
spark.stop()